# گام سوم: تحلیل اکتشافی داده‌ها (EDA) — پروژه تشخیص صدا

**موضوع:** تشخیص و طبقه‌بندی صداهای محیطی با مجموعه‌داده ESC-50  
**هدف:** شناخت ساختار داده، کنترل کیفیت، محاسبه آمار توصیفی، طرح سؤال‌های تحلیلی و تهیه نمودارهای مناسب.

این نوت‌بوک بر اساس ساختار نوت‌بوک آزمایشگاه کلاس (`lab3_nim_minimax_eda.ipynb`) تنظیم شده است؛ با این تفاوت که داده بازی Nim با داده‌های صوتی ESC-50 جایگزین شده‌اند.

## محدوده تحلیل
در این مرحله روی **طبقه‌بندی صداهای محیطی** کار می‌کنیم؛ نه تبدیل گفتار به متن و نه تشخیص هویت گوینده. داده اصلی شامل کلیپ‌های صوتی برچسب‌دار است.

## قبل از اجرا
- نوت‌بوک را از ابتدا تا انتها اجرا کنید.
- اجرای کامل روی ۲۰۰۰ فایل صوتی ممکن است چند دقیقه زمان ببرد.
- اگر اینترنت در محیط Jupyter در دسترس نیست، فایل ZIP رسمی ESC-50 را دستی دریافت و در پوشه `data` استخراج کنید.
- خود فایل‌های WAV را در مخزن عمومی GitHub بارگذاری نکنید؛ مستندات مجوز مجموعه‌داده را رعایت کنید.

**منبع داده:** https://github.com/karolpiczak/ESC-50  
**مقاله:** Piczak (2015), DOI: https://doi.org/10.1145/2733373.2806390


## ۱. آماده‌سازی کتابخانه‌ها و مسیر داده

کتابخانه‌های مورد استفاده:
- `pandas` برای بررسی جدول فراداده
- `numpy` برای محاسبات عددی
- `matplotlib` برای نمودارها
- `scipy` برای خواندن فایل WAV و تحلیل سیگنال

در صورت نبود کتابخانه‌ها، دستور نصب را در یک سلول جداگانه اجرا کنید:
`%pip install numpy pandas matplotlib scipy`


In [ ]:
from pathlib import Path
import os
import zipfile
import urllib.request
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import wavfile

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

# مسیر داده‌ها
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# اگر داده‌ها را قبلاً در مسیر دیگری استخراج کرده‌اید، این متغیر را تغییر دهید.
# مسیر باید پوشه‌ای باشد که داخل آن پوشه audio و پوشه meta قرار دارد.
DATASET_DIR = DATA_DIR / "ESC-50-master"

# برای اجرای سریع‌تر می‌توانید مثلاً عدد 200 را قرار دهید.
# مقدار None یعنی تحلیل همه فایل‌های صوتی موجود.
MAX_AUDIO_FILES = None

print("Python working directory:", Path.cwd())
print("Dataset directory:", DATASET_DIR.resolve())


## ۲. دریافت یا پیدا کردن مجموعه‌داده

کد زیر ابتدا پوشه موجود را بررسی می‌کند. اگر مجموعه‌داده پیدا نشود، نسخه ZIP را از مخزن اصلی دانلود می‌کند. حجم مجموعه‌داده قابل توجه است؛ در صورت کندی اینترنت، آن را با مرورگر دانلود و در `data` استخراج کنید.

ساختار مورد انتظار:
```text
data/
└── ESC-50-master/
    ├── audio/
    ├── meta/
    │   └── esc50.csv
    └── LICENSE
```


In [ ]:
def find_dataset_dir(base_dir: Path):
    """پوشه‌ای را پیدا می‌کند که meta/esc50.csv و audio/ را داشته باشد."""
    candidates = [base_dir, *base_dir.glob("**/ESC-50-master"), *base_dir.glob("**/ESC-50")]
    for candidate in candidates:
        if (candidate / "meta" / "esc50.csv").exists() and (candidate / "audio").exists():
            return candidate
    # جست‌وجوی عمومی برای حالتی که پوشه نام دیگری دارد
    for csv_path in base_dir.glob("**/esc50.csv"):
        candidate = csv_path.parent.parent
        if (candidate / "audio").exists():
            return candidate
    return None

dataset_path = find_dataset_dir(DATA_DIR)

if dataset_path is None:
    zip_path = DATA_DIR / "ESC-50-master.zip"
    download_url = "https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip"
    print("Dataset not found. Starting download from the official repository...")
    try:
        urllib.request.urlretrieve(download_url, zip_path)
        print("Download completed. Extracting...")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(DATA_DIR)
        dataset_path = find_dataset_dir(DATA_DIR)
    except Exception as exc:
        raise RuntimeError(
            "دانلود خودکار موفق نشد. فایل ZIP را از نشانی رسمی "
            "https://github.com/karolpiczak/ESC-50 دریافت کنید، در پوشه data استخراج کنید "
            "و سلول را دوباره اجرا کنید. جزئیات خطا: " + str(exc)
        )

if dataset_path is None:
    raise FileNotFoundError("پوشه داده پیدا نشد؛ مسیر DATASET_DIR را اصلاح کنید.")

DATASET_DIR = dataset_path
AUDIO_DIR = DATASET_DIR / "audio"
META_PATH = DATASET_DIR / "meta" / "esc50.csv"

print("Dataset found at:", DATASET_DIR.resolve())
print("Audio folder exists:", AUDIO_DIR.exists())
print("Metadata file exists:", META_PATH.exists())


## ۳. بارگذاری فراداده و شناخت اولیه جدول

فایل `esc50.csv` مشخصات هر کلیپ را نگهداری می‌کند. ستون‌های اصلی عبارت‌اند از نام فایل، شماره Fold، برچسب عددی، نام کلاس، عضویت در زیرمجموعه ESC-10، شناسه فایل منبع و شناسه برداشت (Take).


In [ ]:
df = pd.read_csv(META_PATH)
metadata_cols = df.columns.tolist()  # ستون‌های اصلی فایل CSV، پیش از افزودن ستون‌های تحلیلی

print("اندازه جدول (ردیف، ستون):", df.shape)
display(df.head())
print("\nنام ستون‌ها:")
print(df.columns.tolist())


## ۴. نوع ویژگی‌ها و کیفیت داده

در این قسمت بررسی می‌کنیم:
- آیا مقدار گمشده وجود دارد؟
- آیا ردیف یا نام فایل تکراری وجود دارد؟
- نوع داده هر ستون چیست؟
- آیا فایل صوتی متناظر با هر ردیف در پوشه `audio` موجود است؟
- آیا تعداد کلاس‌ها و Foldها با ساختار اعلام‌شده مجموعه‌داده سازگار است؟

توجه: تکراری بودن `src_file` به‌تنهایی به معنی خطا نیست؛ چند قطعه ممکن است از یک فایل منبع استخراج شده باشند. در این مجموعه، قطعات یک منبع اصلی برای جلوگیری از نشت داده در یک Fold نگه داشته شده‌اند.


In [ ]:
print("نوع داده ستون‌ها:")
display(df.dtypes.to_frame("dtype"))

missing = df.isna().sum().rename("missing_count")
print("\nمقادیر گمشده در هر ستون:")
display(missing.to_frame())

print("تعداد ردیف‌های کاملاً تکراری:", int(df.duplicated().sum()))
if "filename" in df.columns:
    print("تعداد نام فایل‌های تکراری:", int(df["filename"].duplicated().sum()))

print("\nتعداد مقادیر یکتا در ستون‌های دسته‌ای:")
for col in df.select_dtypes(include=["object", "bool"]).columns:
    print(f"{col}: {df[col].nunique()}")

print("\nتوزیع Foldها:")
display(df["fold"].value_counts().sort_index().rename("samples").to_frame())

print("\nتعداد کلاس‌ها:", df["category"].nunique())
print("تعداد نمونه‌ها:", len(df))
print("تعداد نمونه‌های ESC-10:", int(df["esc10"].sum()) if "esc10" in df.columns else "column not found")


In [ ]:
# بررسی وجود واقعی فایل‌های صوتی
df["audio_path"] = df["filename"].apply(lambda name: AUDIO_DIR / str(name))
df["file_exists"] = df["audio_path"].apply(lambda p: p.exists())

print("فایل‌های صوتی موجود:", int(df["file_exists"].sum()))
print("فایل‌های صوتی مفقود:", int((~df["file_exists"]).sum()))

missing_audio = df.loc[~df["file_exists"], ["filename", "category", "fold"]]
display(missing_audio.head(10))


## ۵. آمار توصیفی فراداده و توازن کلاس‌ها

برای ستون‌های عددی از شاخص‌هایی مانند میانگین، انحراف معیار، کمینه، چارک‌ها و بیشینه استفاده می‌کنیم. برای ستون‌های دسته‌ای نیز تعداد نمونه هر کلاس را می‌سنجیم.

طبق معرفی رسمی ESC-50، این مجموعه ۲۰۰۰ فایل در ۵۰ کلاس دارد و در طراحی آن ۴۰ نمونه برای هر کلاس در نظر گرفته شده است. اجرای سلول بعدی این وضعیت را روی فایل فراداده دانلودشده کنترل می‌کند.


In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
display(df[numeric_cols].describe().T.round(3))

class_counts = df["category"].value_counts().sort_values(ascending=False)
class_summary = pd.DataFrame({
    "sample_count": class_counts,
    "percentage": (class_counts / len(df) * 100).round(2)
})
display(class_summary)

print("کمترین تعداد نمونه در یک کلاس:", int(class_counts.min()))
print("بیشترین تعداد نمونه در یک کلاس:", int(class_counts.max()))
print("آیا همه کلاس‌ها دقیقاً ۴۰ نمونه دارند؟", bool((class_counts == 40).all()))


## ۶. تعریف پنج گروه صوتی اصلی

برای مقایسه سطح کلی‌تر، کلاس‌های ۵۰گانه را به پنج گروه معنایی مجموعه ESC-50 نگاشت می‌کنیم: حیوانات، صداهای طبیعی و آب، صداهای انسانی غیرگفتاری، صداهای داخل ساختمان و صداهای بیرونی/شهری.

این نگاشت صرفاً برای تحلیل و نمایش است و جای ستون اصلی `category` را نمی‌گیرد.


In [ ]:
major_groups = {
    "animals": [
        "dog", "rooster", "pig", "cow", "frog", "cat", "hen",
        "insects", "sheep", "crow"
    ],
    "natural": [
        "rain", "sea_waves", "crackling_fire", "crickets", "chirping_birds",
        "water_drops", "wind", "pouring_water", "toilet_flush", "thunderstorm"
    ],
    "human_non_speech": [
        "crying_baby", "sneezing", "clapping", "breathing", "coughing",
        "footsteps", "laughing", "brushing_teeth", "snoring", "drinking_sipping"
    ],
    "interior": [
        "door_wood_knock", "mouse_click", "keyboard_typing", "door_wood_creaks",
        "can_opening", "washing_machine", "vacuum_cleaner", "clock_alarm",
        "clock_tick", "glass_breaking"
    ],
    "exterior_urban": [
        "helicopter", "chainsaw", "siren", "car_horn", "engine",
        "train", "church_bells", "airplane", "fireworks", "hand_saw"
    ]
}

category_to_major = {
    category: group
    for group, categories in major_groups.items()
    for category in categories
}
df["major_category"] = df["category"].map(category_to_major)

unmapped = sorted(df.loc[df["major_category"].isna(), "category"].unique())
print("کلاس‌های بدون گروه:", unmapped if unmapped else "هیچ‌کدام")
display(df["major_category"].value_counts(dropna=False).to_frame("sample_count"))


## ۷. استخراج ویژگی‌های پایه از سیگنال صوتی

هر فایل صوتی علاوه بر برچسب، یک سیگنال عددی دارد. برای شناخت اولیه سیگنال، ویژگی‌های زیر استخراج می‌شوند:

- **Duration:** طول فایل بر حسب ثانیه
- **Sample rate:** تعداد نمونه در هر ثانیه
- **Channels:** تعداد کانال صوتی
- **Peak amplitude:** بیشترین دامنه مطلق سیگنال نرمال‌شده
- **RMS:** مقدار مؤثر دامنه؛ شاخصی تقریبی از انرژی/بلندی نسبی
- **Zero-crossing rate:** نرخ عبور سیگنال از صفر
- **Spectral centroid:** مرکز طیفی؛ شاخصی تقریبی از روشنایی/توزیع فرکانسی صدا

این ویژگی‌ها توصیفی‌اند؛ RMS یا مرکز طیفی به‌تنهایی معادل بلندی ادراکی یا برچسب کلاس نیستند.


In [ ]:
def to_float_audio(signal):
    """تبدیل سیگنال عددی WAV به float در بازه تقریبی [-1, 1]."""
    if signal.ndim == 2:
        signal = signal.mean(axis=1)
    if np.issubdtype(signal.dtype, np.integer):
        info = np.iinfo(signal.dtype)
        scale = max(abs(info.min), abs(info.max))
        signal = signal.astype(np.float32) / float(scale)
    else:
        signal = signal.astype(np.float32)
    return signal

def spectral_centroid(signal, sr, frame_length=2048, hop_length=1024):
    """محاسبه مرکز طیفی میانگین با پنجره‌های کوتاه و FFT."""
    if len(signal) < frame_length:
        signal = np.pad(signal, (0, frame_length - len(signal)))
    window = np.hanning(frame_length).astype(np.float32)
    centroids = []
    freqs = np.fft.rfftfreq(frame_length, d=1.0 / sr)
    for start in range(0, max(1, len(signal) - frame_length + 1), hop_length):
        frame = signal[start:start + frame_length]
        if len(frame) < frame_length:
            frame = np.pad(frame, (0, frame_length - len(frame)))
        magnitude = np.abs(np.fft.rfft(frame * window))
        total = magnitude.sum()
        if total > 0:
            centroids.append(float(np.sum(freqs * magnitude) / total))
    return float(np.mean(centroids)) if centroids else 0.0

def extract_audio_features(path):
    sr, raw = wavfile.read(path)
    signal = to_float_audio(raw)
    duration = len(signal) / float(sr) if sr else np.nan
    peak = float(np.max(np.abs(signal))) if len(signal) else np.nan
    rms = float(np.sqrt(np.mean(signal ** 2))) if len(signal) else np.nan
    zcr = float(np.mean(signal[:-1] * signal[1:] < 0)) if len(signal) > 1 else np.nan
    centroid = spectral_centroid(signal, sr) if len(signal) and sr else np.nan
    return {
        "duration_sec": duration,
        "sample_rate_hz": int(sr),
        "channels": int(raw.shape[1]) if raw.ndim == 2 else 1,
        "bit_depth": int(raw.dtype.itemsize * 8),
        "peak_amplitude": peak,
        "rms": rms,
        "zero_crossing_rate": zcr,
        "spectral_centroid_hz": centroid
    }

audio_rows = []
audio_files = df.loc[df["file_exists"]].copy()
if MAX_AUDIO_FILES is not None:
    audio_files = audio_files.sample(min(MAX_AUDIO_FILES, len(audio_files)), random_state=RANDOM_STATE)

errors = []
for row in audio_files.itertuples(index=False):
    try:
        features = extract_audio_features(row.audio_path)
        features.update({
            "filename": row.filename,
            "category": row.category,
            "major_category": row.major_category,
            "fold": row.fold
        })
        audio_rows.append(features)
    except Exception as exc:
        errors.append({"filename": row.filename, "error": str(exc)})

audio_df = pd.DataFrame(audio_rows)
print("تعداد فایل‌های تحلیل‌شده:", len(audio_df))
print("تعداد فایل‌های دارای خطا:", len(errors))
if errors:
    display(pd.DataFrame(errors).head())
display(audio_df.head())


## ۸. آمار توصیفی ویژگی‌های صوتی

در این قسمت میانه، میانگین، انحراف معیار، چارک‌ها و دامنه ویژگی‌های عددی صوتی را بررسی می‌کنیم. اگر همه فایل‌های مجموعه تحلیل شده باشند، آمار برای کل دیتاست است؛ در غیر این صورت فقط برای نمونه انتخاب‌شده معتبر است.


In [ ]:
audio_numeric = [
    "duration_sec", "sample_rate_hz", "channels", "bit_depth",
    "peak_amplitude", "rms", "zero_crossing_rate", "spectral_centroid_hz"
]
display(audio_df[audio_numeric].describe().T.round(4))

print("تعداد نمونه‌های قابل تحلیل:", len(audio_df))
print("آیا تحلیل روی همه فایل‌های موجود انجام شده؟", len(audio_df) == int(df["file_exists"].sum()))


## ۹. سه سؤال تحلیلی پروژه (چه باید دید؟)

**سؤال ۱ — توازن کلاس‌ها:** آیا تعداد نمونه‌ها در کلاس‌های مختلف تقریباً یکسان است یا بعضی کلاس‌ها نمونه بیشتری دارند؟  
شاخص/نمودار مناسب: تعداد نمونه در هر کلاس و نمودار میله‌ای افقی.

**سؤال ۲ — کیفیت و یکنواختی فایل‌ها:** آیا فایل‌ها از نظر مدت، نرخ نمونه‌برداری، تعداد کانال و عمق بیت با هم سازگارند؟  
شاخص/نمودار مناسب: کمینه، بیشینه و انحراف معیار ویژگی‌ها؛ هیستوگرام مدت فایل و نمودار توزیع نرخ نمونه‌برداری.

**سؤال ۳ — تفاوت ویژگی‌های صوتی بین گروه‌ها:** آیا گروه‌های صوتی از نظر انرژی نسبی (RMS) یا مرکز طیفی تفاوت توصیفی دارند؟  
شاخص/نمودار مناسب: میانگین و میانه ویژگی‌ها به تفکیک گروه، نمودار Boxplot و ماتریس همبستگی.

این سؤال‌ها اکتشافی‌اند؛ پاسخ آن‌ها به‌تنهایی عملکرد یک مدل یادگیری ماشین را ثابت نمی‌کند.


In [ ]:
# پاسخ عددی خودکار به سؤال ۱
print("سؤال ۱: توازن کلاس‌ها")
print("تعداد کلاس‌ها:", df["category"].nunique())
print("کمترین/بیشترین تعداد نمونه در کلاس:", int(class_counts.min()), "/", int(class_counts.max()))
print("تعداد کلاس‌هایی با ۴۰ نمونه:", int((class_counts == 40).sum()), "از", len(class_counts))

# پاسخ عددی خودکار به سؤال ۲
print("\nسؤال ۲: سازگاری مشخصات فنی فایل‌ها")
if not audio_df.empty:
    for col in ["duration_sec", "sample_rate_hz", "channels", "bit_depth"]:
        print(f"{col}: min={audio_df[col].min()}, max={audio_df[col].max()}, unique={audio_df[col].nunique()}")
else:
    print("داده صوتی برای بررسی وجود ندارد.")

# پاسخ عددی خودکار به سؤال ۳
print("\nسؤال ۳: مقایسه ویژگی‌های صوتی بین گروه‌ها")
if not audio_df.empty:
    display(audio_df.groupby("major_category")[["rms", "spectral_centroid_hz"]]
            .agg(["mean", "median", "std"]).round(4))
else:
    print("داده صوتی برای مقایسه وجود ندارد.")


## ۱۰. نمودار اول: تعداد نمونه در هر کلاس

این نمودار به سؤال اول پاسخ می‌دهد. در یک مجموعه متوازن، ارتفاع میله‌ها تقریباً برابر خواهد بود. اگر عدم توازن دیده شود، در مرحله مدل‌سازی باید اثر آن روی معیارهای ارزیابی بررسی شود.


In [ ]:
plt.figure(figsize=(12, 12))
class_counts.sort_values().plot(kind="barh")
plt.title("Number of audio clips per class (ESC-50)")
plt.xlabel("Number of clips")
plt.ylabel("Sound class")
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()


## ۱۱. نمودار دوم: توزیع مدت فایل‌ها و نرخ نمونه‌برداری

این نمودارها برای شناسایی فایل‌های غیرعادی یا ناسازگاری فنی به کار می‌روند. اگر همه کلیپ‌ها در مدت و نرخ نمونه‌برداری یکسان باشند، توزیع به یک مقدار ثابت نزدیک می‌شود.


In [ ]:
if not audio_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(audio_df["duration_sec"].dropna(), bins=20, edgecolor="black")
    axes[0].set_title("Audio duration distribution")
    axes[0].set_xlabel("Duration (seconds)")
    axes[0].set_ylabel("Number of files")

    sr_counts = audio_df["sample_rate_hz"].value_counts().sort_index()
    axes[1].bar(sr_counts.index.astype(str), sr_counts.values)
    axes[1].set_title("Sample-rate distribution")
    axes[1].set_xlabel("Sample rate (Hz)")
    axes[1].set_ylabel("Number of files")
    plt.tight_layout()
    plt.show()
else:
    print("ابتدا سلول استخراج ویژگی صوتی را اجرا کنید.")


## ۱۲. نمودار سوم: شکل موج و طیف‌نگاره یک نمونه

شکل موج، تغییر دامنه سیگنال را در طول زمان نشان می‌دهد. طیف‌نگاره، انرژی فرکانس‌ها را در طول زمان نمایش می‌دهد و برای شناخت ساختار زمانی-فرکانسی صدا مفید است.

نمونه به‌صورت تصادفی انتخاب می‌شود؛ نام کلاس آن در عنوان نمودار درج خواهد شد.


In [ ]:
if len(audio_df) > 0:
    sample_row = audio_df.sample(1, random_state=RANDOM_STATE).iloc[0]
    sample_path = AUDIO_DIR / sample_row["filename"]
    sr, raw = wavfile.read(sample_path)
    sig = to_float_audio(raw)
    time_axis = np.arange(len(sig)) / sr

    fig, axes = plt.subplots(2, 1, figsize=(13, 7))
    axes[0].plot(time_axis, sig, linewidth=0.6)
    axes[0].set_title(f"Waveform — {sample_row['category']} ({sample_row['filename']})")
    axes[0].set_xlabel("Time (seconds)")
    axes[0].set_ylabel("Amplitude")
    axes[0].grid(alpha=0.2)

    axes[1].specgram(sig, NFFT=1024, Fs=sr, noverlap=768, cmap="magma")
    axes[1].set_title("Spectrogram")
    axes[1].set_xlabel("Time (seconds)")
    axes[1].set_ylabel("Frequency (Hz)")
    plt.tight_layout()
    plt.show()

    try:
        from IPython.display import Audio, display
        display(Audio(sig, rate=sr))
    except Exception as exc:
        print("پخش صوت در این محیط در دسترس نیست:", exc)
else:
    print("فایل صوتی قابل نمایش پیدا نشد.")


## ۱۳. نمودار چهارم: مقایسه RMS بین گروه‌های صوتی

RMS شاخصی از دامنه مؤثر سیگنال است. Boxplot پراکندگی و میانه را نشان می‌دهد و برای مقایسه گروه‌ها مناسب است. تفاوت RMS ممکن است به شدت ضبط، فاصله منبع از میکروفون یا ماهیت صدا مربوط باشد؛ بنابراین نباید آن را مستقیماً معادل بلندی ادراکی دانست.


In [ ]:
if not audio_df.empty:
    order = (audio_df.groupby("major_category")["rms"].median()
             .sort_values().index.tolist())
    data = [audio_df.loc[audio_df["major_category"] == g, "rms"].dropna().values
            for g in order]
    plt.figure(figsize=(10, 5))
    plt.boxplot(data, labels=order, showfliers=True)
    plt.title("RMS distribution by major sound group")
    plt.ylabel("RMS amplitude")
    plt.xlabel("Major sound group")
    plt.xticks(rotation=20, ha="right")
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()
else:
    print("داده صوتی برای رسم نمودار وجود ندارد.")


## ۱۴. نمودار پنجم: مرکز طیفی بین گروه‌های صوتی

مرکز طیفی، مرکز ثقل طیف دامنه است. مقدار بالاتر می‌تواند نشان‌دهنده سهم بیشتر فرکانس‌های بالاتر باشد، اما این ویژگی به‌تنهایی نوع صدا را تعیین نمی‌کند.


In [ ]:
if not audio_df.empty:
    order = (audio_df.groupby("major_category")["spectral_centroid_hz"].median()
             .sort_values().index.tolist())
    data = [audio_df.loc[audio_df["major_category"] == g, "spectral_centroid_hz"].dropna().values
            for g in order]
    plt.figure(figsize=(10, 5))
    plt.boxplot(data, labels=order, showfliers=True)
    plt.title("Spectral centroid distribution by major sound group")
    plt.ylabel("Spectral centroid (Hz)")
    plt.xlabel("Major sound group")
    plt.xticks(rotation=20, ha="right")
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()
else:
    print("داده صوتی برای رسم نمودار وجود ندارد.")


## ۱۵. نمودار ششم: همبستگی ویژگی‌های عددی صوت

ماتریس همبستگی رابطه خطی بین ویژگی‌های عددی را نشان می‌دهد. همبستگی بالا به معنی رابطه علت و معلولی نیست؛ فقط نشان می‌دهد دو ویژگی در این داده‌ها چگونه با هم تغییر می‌کنند.


In [ ]:
if len(audio_df) > 1:
    corr_cols = ["duration_sec", "sample_rate_hz", "peak_amplitude",
                 "rms", "zero_crossing_rate", "spectral_centroid_hz"]
    corr = audio_df[corr_cols].corr(numeric_only=True)

    fig, ax = plt.subplots(figsize=(8, 6))
    image = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
    ax.set_xticks(range(len(corr_cols)), corr_cols, rotation=45, ha="right")
    ax.set_yticks(range(len(corr_cols)), corr_cols)
    ax.set_title("Correlation matrix of audio features")
    for i in range(len(corr_cols)):
        for j in range(len(corr_cols)):
            ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
    fig.colorbar(image, ax=ax, label="Correlation")
    plt.tight_layout()
    plt.show()
else:
    print("برای محاسبه همبستگی، حداقل دو فایل لازم است.")


## ۱۶. پاسخ به سؤال‌های تحلیلی با الگوی مشاهده، دلیل، نتیجه

پس از اجرای سلول زیر، مقادیر واقعی در خروجی نمایش داده می‌شوند. در گزارش نهایی، برای هر سؤال سه جمله بنویسید:
1. **مشاهده:** چه عدد یا الگویی در جدول/نمودار دیده شد؟
2. **دلیل احتمالی:** چه ویژگی داده یا شرایط ضبط می‌تواند آن را توضیح دهد؟
3. **نتیجه:** این موضوع چه اثری بر آماده‌سازی داده یا مرحله مدل‌سازی دارد؟

از نوشتن برداشت قطعی درباره تفاوت شنیداری کلاس‌ها فقط بر اساس یک ویژگی عددی خودداری کنید.


In [ ]:
# تولید چند جمله عددی برای کمک به نوشتن تفسیر
print("=== جمع‌بندی خودکار EDA ===")
print(f"تعداد ردیف‌های فراداده: {len(df)}")
print(f"تعداد کلاس‌ها: {df['category'].nunique()}")
print(f"تعداد Foldها: {df['fold'].nunique()}")
print(f"مقادیر گمشده در فراداده: {int(df.isna().sum().sum())}")
print(f"ردیف‌های تکراری: {int(df.duplicated().sum())}")
print(f"فایل‌های صوتی پیدا نشده: {int((~df['file_exists']).sum())}")

if len(class_counts):
    print(f"دامنه تعداد نمونه هر کلاس: {class_counts.min()} تا {class_counts.max()}")

if not audio_df.empty:
    print(f"دامنه مدت فایل‌ها: {audio_df['duration_sec'].min():.3f} تا {audio_df['duration_sec'].max():.3f} ثانیه")
    print("نرخ‌های نمونه‌برداری مشاهده‌شده:", sorted(audio_df["sample_rate_hz"].dropna().unique().tolist()))
    print("تعداد کانال‌های مشاهده‌شده:", sorted(audio_df["channels"].dropna().unique().tolist()))
    group_rms = audio_df.groupby("major_category")["rms"].mean().sort_values(ascending=False)
    print("بالاترین میانگین RMS:", group_rms.index[0], f"({group_rms.iloc[0]:.4f})")
    group_centroid = audio_df.groupby("major_category")["spectral_centroid_hz"].mean().sort_values(ascending=False)
    print("بالاترین مرکز طیفی میانگین:", group_centroid.index[0], f"({group_centroid.iloc[0]:.1f} Hz)")


## ۱۷. ذخیره خروجی‌های تحلیل

برای مستندسازی مرحله EDA، جدول ویژگی‌های استخراج‌شده و خلاصه کلاس‌ها به‌صورت CSV ذخیره می‌شوند. این فایل‌ها خروجی تحلیل هستند و می‌توان آن‌ها را در مخزن GitHub قرار داد؛ خود فایل‌های صوتی را به مخزن اضافه نکنید.


In [ ]:
features_output = Path("eda_audio_features.csv")
class_output = Path("eda_class_summary.csv")
quality_output = Path("eda_quality_summary.csv")

audio_df.to_csv(features_output, index=False, encoding="utf-8-sig")
class_summary.to_csv(class_output, encoding="utf-8-sig")

quality_summary = pd.DataFrame({
    "check": [
        "metadata_rows",
        "metadata_columns",
        "missing_metadata_cells",
        "duplicate_metadata_rows",
        "duplicate_filenames",
        "audio_files_found",
        "audio_files_missing",
        "audio_files_analysed"
    ],
    "value": [
        len(df),
        len(df.columns) - 2,  # audio_path و file_exists در جدول کاری اضافه شده‌اند
        int(df[metadata_cols].isna().sum().sum()),
        int(df.duplicated(subset=metadata_cols).sum()),
        int(df["filename"].duplicated().sum()),
        int(df["file_exists"].sum()),
        int((~df["file_exists"]).sum()),
        len(audio_df)
    ]
})
quality_summary.to_csv(quality_output, index=False, encoding="utf-8-sig")

print("فایل‌های خروجی ذخیره شدند:")
print(features_output.resolve())
print(class_output.resolve())
print(quality_output.resolve())
display(quality_summary)


## ۱۸. نتیجه‌گیری و محدودیت‌های تحلیل

### جمع‌بندی
این EDA ساختار فراداده و کیفیت فایل‌های صوتی را بررسی می‌کند، توازن کلاس‌ها را می‌سنجد و چند ویژگی پایه سیگنال را برای مقایسه گروه‌ها استخراج می‌کند. خروجی‌های عددی نهایی پس از اجرای کامل نوت‌بوک به دست می‌آیند.

### محدودیت‌ها
- ESC-50 فقط ۵۰ کلاس صدای محیطی را پوشش می‌دهد و نماینده تمام صداهای واقعی نیست.
- RMS و مرکز طیفی ویژگی‌های ساده‌اند و برای تشخیص قطعی کلاس کافی نیستند.
- تفاوت میکروفون، فاصله، محیط ضبط و نویز می‌تواند روی ویژگی‌ها اثر بگذارد.
- برای ارزیابی مدل باید تقسیم‌بندی Foldهای اصلی رعایت شود و داده‌های یک منبع اصلی نباید بین آموزش و آزمون نشت کنند.
- EDA به‌تنهایی دقت مدل را نشان نمی‌دهد؛ ارزیابی مدل در مرحله بعد و با معیارهایی مانند Precision، Recall و F1-score انجام می‌شود.

### گام بعدی
پس از تأیید سلامت داده و شناخت توزیع کلاس‌ها، مرحله آماده‌سازی ورودی مدل و ساخت ویژگی‌های مناسب‌تر (مانند Mel-spectrogram یا MFCC) انجام می‌شود.


## منابع
1. Piczak, K. J. (2015). *ESC: Dataset for Environmental Sound Classification*. Proceedings of the 23rd ACM International Conference on Multimedia. https://doi.org/10.1145/2733373.2806390
2. مخزن اصلی ESC-50: https://github.com/karolpiczak/ESC-50
3. مشخصات مجموعه‌داده و مجوز: https://audeering.github.io/datasets/datasets/esc-50.html
